# P3. Дедупликация, P4. Сборка сюжетов

Вход: `data/clean/news_2016.csv` (P1), `data/embeddings/matrix.npy` и `id_map.csv` (P2, строка матрицы = строка CSV, L2-норма).

Выход:
- `data/dedup/pairs.csv` - пары дубликатов: статьи, уровень, score
- `data/stories/story_map.csv` - `article_id -> story_id`, размер сюжета, is_canonical
- `data/stories/canonical.csv` - канонические документы, вход P5
- `data/reports/p3_levels.csv`, `p4_story_sizes.csv`

Уровни дедупликации:
1. sha256 нормализованного текста, точные копии (P3)
2. MinHash + LSH, Jaccard >= 0.8 (P3)
3. косинус >= 0.90 в окне +-3 дня (P4)

Все пары объединяются в граф, компоненты связности = сюжеты.

In [ ]:
import hashlib
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from datasketch import MinHash, MinHashLSH
from tqdm.auto import tqdm

DATA = Path.cwd().parent / "data"
CLEAN = DATA / "clean" / "news_2016.csv"
EMB = DATA / "embeddings"
DEDUP = DATA / "dedup"
STORIES = DATA / "stories"
REPORTS = DATA / "reports"

# P3
SHINGLE = 5  # слов в шингле
NUM_PERM = 128
JACCARD_MIN = 0.8

# P4
COS_MIN = 0.90
WINDOW_DAYS = 3  # +- дней
BLOCK = 2048

# резка больших компонент
MAX_STORY = 60
COS_STEP = 0.01
COS_CAP = 0.98

CANONICAL = "earliest"  # earliest | longest

PAIR_COLS = ["row_a", "row_b", "level", "score"]

# стиль графиков
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": "#e8e7e3", "grid.linewidth": 0.8,
    "figure.facecolor": "white", "axes.titlelocation": "left",
})
SERIES = {"exact": "#2a78d6", "minhash": "#eb6834", "semantic": "#1baf7a"}

## Нормализация для сравнения

Для уровней P3 сравниваются слова текста без регистра и пунктуации (набор знаков как в P1).

In [ ]:
PUNCT = str.maketrans({c: " " for c in "«»\"'`.,;:!?()[]{}—–-…%№*/\\|+=<>@#$^&~"})

def words(text):
    """Слова текста без регистра и пунктуации."""
    return text.lower().translate(PUNCT).split()

def text_hash(ws):
    """sha256 нормализованного текста (уровень 1)."""
    return hashlib.sha256(" ".join(ws).encode()).digest()

def shingles(ws):
    """Шинглы по SHINGLE слов, в bytes для datasketch."""
    return {" ".join(ws[i:i + SHINGLE]).encode()
            for i in range(len(ws) - SHINGLE + 1)}

def jaccard(a, b):
    """Коэффициент Жаккара."""
    common = len(a & b)
    return common / (len(a) + len(b) - common)

## Корпус

In [ ]:
t0 = time.perf_counter()
df = pd.read_csv(CLEAN, dtype="string")
df["dt_utc"] = pd.to_datetime(df["dt_utc"], utc=True, format="ISO8601")
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")

# article_id по номеру строки, как в embed.py
aid = np.array([f"a_{i:07d}" for i in range(len(df))], dtype=object)
df["article_id"] = aid

print(f"статей: {len(df)}, прочитано за {time.perf_counter() - t0:.0f} c")
print(f"период: {df['dt_utc'].min():%Y-%m-%d} — {df['dt_utc'].max():%Y-%m-%d}")
df.head(3)

## P3. Уровень 1: точные дубликаты

Группировка по хэшу. В pairs пишется звезда от первой статьи группы (компоненты те же, рёбер меньше).
Уровни 2 и 3 дальше работают только по первым статьям групп (`uniq`).

In [ ]:
groups = defaultdict(list)
for row, text in enumerate(tqdm(df["text"], desc="хэши")):
    ws = words(text)
    if ws:
        groups[text_hash(ws)].append(row)

ea, eb = [], []
for g in groups.values():
    ea.extend([g[0]] * (len(g) - 1))
    eb.extend(g[1:])
exact = pd.DataFrame({"row_a": ea, "row_b": eb, "level": "exact", "score": 1.0})

uniq = np.array(sorted(g[0] for g in groups.values()), dtype=np.int64)

print(f"групп по хэшу:  {len(groups)}")
print(f"точных пар:     {len(exact)}")
print(f"уникальных:     {len(uniq)} из {len(df)} ({len(uniq) / len(df):.1%})")

## P3. Уровень 2: MinHash + LSH

Кандидаты через LSH (число бэндов datasketch подбирает по порогу), затем точный Jaccard по кандидатам.
Шинглы не храним, для кандидатов считаются заново.

In [ ]:
# copy() прототипа, чтобы перестановки не создавались для каждой сигнатуры
proto = MinHash(num_perm=NUM_PERM)
is_uniq = np.zeros(len(df), dtype=bool)
is_uniq[uniq] = True

sigs = {}
for row, text in enumerate(tqdm(df["text"], desc="MinHash")):
    if not is_uniq[row]:
        continue
    sh = shingles(words(text))
    if not sh:
        continue  # текст короче шингла
    m = proto.copy()
    m.update_batch(sh)
    sigs[row] = m

print(f"сигнатур: {len(sigs)}")

In [ ]:
lsh = MinHashLSH(threshold=JACCARD_MIN, num_perm=NUM_PERM)
with lsh.insertion_session() as session:
    for row, m in tqdm(sigs.items(), desc="LSH: вставка"):
        session.insert(row, m)

candidates = set()
for row, m in tqdm(sigs.items(), desc="LSH: запросы"):
    for other in lsh.query(m):
        if other != row:
            candidates.add((row, other) if row < other else (other, row))

del sigs, lsh
print(f"пар-кандидатов: {len(candidates)}")

In [ ]:
# точный Jaccard для кандидатов
need = sorted({row for pair in candidates for row in pair})
sh_of = {row: shingles(words(df["text"].iat[row]))
         for row in tqdm(need, desc="шинглы кандидатов")}

checked = [(a, b, jaccard(sh_of[a], sh_of[b])) for a, b in tqdm(candidates, desc="Жаккар")]
del sh_of

minhash = pd.DataFrame(checked, columns=["row_a", "row_b", "score"])
minhash["level"] = "minhash"
minhash = minhash.loc[minhash["score"] >= JACCARD_MIN, PAIR_COLS].round({"score": 4})

print(f"MinHash-пар: {len(minhash)}, отсеяно кандидатов: {len(checked) - len(minhash)}")

## P3. Карта дубликатов

In [ ]:
def save_pairs(pairs, path):
    """Сохраняет пары с article_id, row_a/row_b (номера строк корпуса и матрицы) тоже остаются."""
    path.parent.mkdir(parents=True, exist_ok=True)
    out = pairs.reset_index(drop=True)
    out.insert(0, "article_id_a", aid[out["row_a"].to_numpy()])
    out.insert(1, "article_id_b", aid[out["row_b"].to_numpy()])
    out.to_csv(path, index=False)
    return out


def level_stats(pairs):
    """Число пар и затронутых статей по уровням."""
    rows = []
    for level, part in pairs.groupby("level", sort=False):
        touched = len(set(part["row_a"]) | set(part["row_b"]))
        rows.append({"уровень": level, "пар": len(part), "статей затронуто": touched,
                     "доля корпуса, %": round(touched / len(df) * 100, 2)})
    return pd.DataFrame(rows).set_index("уровень")


pairs = pd.concat([exact, minhash], ignore_index=True)[PAIR_COLS]
save_pairs(pairs, DEDUP / "pairs.csv")

REPORTS.mkdir(parents=True, exist_ok=True)
p3_stats = level_stats(pairs)
p3_stats.to_csv(REPORTS / "p3_levels.csv")
print(p3_stats.to_string())

### Артефакты P3

Гистограмма Jaccard у кандидатов, для проверки порога 0.8.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = p3_stats["пар"]
axes[0].bar(counts.index, counts.to_numpy(),
            color=[SERIES[i] for i in counts.index], width=0.5)
axes[0].set_title("Найдено пар по уровням")
axes[0].set_ylabel("пар")
for x, v in enumerate(counts.to_numpy()):
    axes[0].text(x, v, f"{v:,}".replace(",", " "), ha="center", va="bottom")

scores = np.array([j for _, _, j in checked])
axes[1].hist(scores, bins=50, range=(0, 1), color=SERIES["minhash"])
axes[1].axvline(JACCARD_MIN, color="#52514e", linewidth=1.5, linestyle="--")
axes[1].text(JACCARD_MIN, axes[1].get_ylim()[1], f" порог {JACCARD_MIN}",
             va="top", color="#52514e")
axes[1].set_title("Жаккар у пар-кандидатов")
axes[1].set_xlabel("J")
axes[1].set_ylabel("пар")

plt.tight_layout()
plt.show()

In [ ]:
def show(row):
    return (f"{df['dt_utc'].iat[row]:%m-%d %H:%M}  {df['source'].iat[row]:<12}"
            f" {df['title'].iat[row][:80]}")


for level, part in pairs.groupby("level", sort=False):
    print(f"=== {level}: {len(part)} пар ===")
    for r in part.sample(min(3, len(part)), random_state=0).itertuples():
        print(f"  score {r.score}")
        print(f"    {show(r.row_a)}")
        print(f"    {show(r.row_b)}")
    print()

## P4. Семантический уровень

Проверка, что матрица и id_map соответствуют текущему корпусу.

In [ ]:
matrix = np.load(EMB / "matrix.npy", mmap_mode="r")
id_map = pd.read_csv(EMB / "id_map.csv")

if len(matrix) != len(df) or len(id_map) != len(df):
    raise ValueError(f"матрица {len(matrix)} и id_map {len(id_map)} против корпуса "
                     f"{len(df)} — P2 считался по другому файлу, нужен пересчёт")
if not (id_map["article_id"].to_numpy() == aid).all():
    raise ValueError("id_map не совпадает с корпусом — нужен пересчёт P2")

print(f"матрица: {matrix.shape} {matrix.dtype}, "
      f"{matrix.nbytes / 2**20:.0f} МБ на диске")
print(f"норма первых векторов: {np.linalg.norm(matrix[:5], axis=1).round(3)}")

Косинус считается в окне +-3 дня. Статьи сортируются по дате, границы окна через searchsorted, по блокам одно матричное умножение, берётся верхний треугольник.
Векторы нормированы, косинус = скалярное произведение.

In [ ]:
ts = df["dt_utc"].to_numpy(dtype="datetime64[ns]").astype("int64")
WINDOW_NS = WINDOW_DAYS * 86_400 * 1_000_000_000
# после fix_dates.py корпус не отсортирован по дате, сортируем uniq по времени
by_time = uniq[np.argsort(ts[uniq], kind="stable")]
u_ts = ts[by_time]

sem_a, sem_b, sem_s = [], [], []
t0 = time.perf_counter()

for start in tqdm(range(0, len(uniq), BLOCK), desc="kNN"):
    stop = min(start + BLOCK, len(uniq))
    # правая граница по последней статье блока; более ранние пары уже посчитаны
    hi = int(np.searchsorted(u_ts, u_ts[stop - 1] + WINDOW_NS, side="right"))

    block = matrix[by_time[start:stop]]
    window = matrix[by_time[start:hi]]
    sim = block @ window.T

    upper = np.arange(start, hi)[None, :] > np.arange(start, stop)[:, None]
    p, q = np.nonzero((sim >= COS_MIN) & upper)
    # точная проверка окна для каждой пары
    fits = np.abs(u_ts[start + q] - u_ts[start + p]) <= WINDOW_NS
    p, q = p[fits], q[fits]

    sem_a.append(by_time[start + p])
    sem_b.append(by_time[start + q])
    sem_s.append(sim[p, q])

semantic = pd.DataFrame({
    "row_a": np.concatenate(sem_a) if sem_a else np.empty(0, np.int64),
    "row_b": np.concatenate(sem_b) if sem_b else np.empty(0, np.int64),
    "level": "semantic",
    "score": np.round(np.concatenate(sem_s) if sem_s else np.empty(0, np.float32), 4),
})[PAIR_COLS]
del sem_a, sem_b, sem_s

print(f"семантических пар: {len(semantic)} за {time.perf_counter() - t0:.0f} c")

In [ ]:
pairs = pd.concat([exact, minhash, semantic], ignore_index=True)[PAIR_COLS]
save_pairs(pairs, DEDUP / "pairs.csv")

p3_stats = level_stats(pairs)
p3_stats.to_csv(REPORTS / "p3_levels.csv")
print(p3_stats.to_string())

## P4. Граф и компоненты связности

Вершины - все статьи (без связей = сюжет из одной статьи), рёбра - пары всех уровней.

Компоненты больше `MAX_STORY` режутся рекурсивно: порог косинуса повышается на `COS_STEP` до `COS_CAP`, удаляются только semantic рёбра.
Число компонент, оставшихся большими после резки, выводится отдельно.

In [ ]:
G = nx.from_pandas_edgelist(pairs, "row_a", "row_b", edge_attr=["level", "score"])
G.add_nodes_from(range(len(df)))
print(f"вершин: {G.number_of_nodes()}, рёбер: {G.number_of_edges()}")


def split(sub, thr):
    """Рекурсивная резка компоненты повышением порога косинуса (до MAX_STORY или COS_CAP)."""
    if sub.number_of_nodes() <= MAX_STORY or thr >= COS_CAP:
        return [set(sub)]
    thr = round(thr + COS_STEP, 4)
    cut = sub.copy()
    cut.remove_edges_from([(u, v) for u, v, d in cut.edges(data=True)
                           if d["level"] == "semantic" and d["score"] < thr])
    return [part for group in nx.connected_components(cut)
            for part in split(cut.subgraph(group), thr)]


raw = list(nx.connected_components(G))
oversized = sum(len(g) > MAX_STORY for g in raw)

components = []
for group in tqdm(raw, desc="компоненты"):
    if len(group) <= MAX_STORY:
        components.append(group)
    else:
        components.extend(split(G.subgraph(group), COS_MIN))

# компоненты, которые не удалось разрезать
stuck = sum(len(g) > MAX_STORY for g in components)

print(f"компонент до резки:  {len(raw)}, из них разросшихся: {oversized}")
print(f"компонент после:     {len(components)}, осталось крупнее {MAX_STORY}: {stuck}")
print(f"крупнейшая: {max(map(len, raw))} -> {max(map(len, components))} статей")

## P4. Канонический документ

Один документ на сюжет для P5. `CANONICAL`: `earliest` - первая публикация, `longest` - самый длинный текст. Остальные ключи сортировки для детерминированности.

In [ ]:
# нумерация сюжетов по самой ранней статье (дата, номер строки)
components.sort(key=lambda group: min((ts[row], row) for row in group))
story_of = np.empty(len(df), dtype=np.int64)
for k, group in enumerate(components):
    for row in group:
        story_of[row] = k

sm = pd.DataFrame({
    "article_id": aid,
    "row": np.arange(len(df)),
    "source": df["source"],
    "dt_utc": df["dt_utc"],
    "story_id": [f"s_{k:06d}" for k in story_of],
    "text_len": df["text"].str.len().astype("int64"),
})
sm["story_size"] = sm.groupby("story_id")["row"].transform("size")

keys, asc = {
    "earliest": (["story_id", "dt_utc", "text_len", "row"], [True, True, False, True]),
    "longest": (["story_id", "text_len", "dt_utc", "row"], [True, False, True, True]),
}[CANONICAL]
sm = sm.sort_values(keys, ascending=asc, kind="stable")
sm["is_canonical"] = ~sm["story_id"].duplicated()
sm = sm.sort_values("row").reset_index(drop=True)

STORIES.mkdir(parents=True, exist_ok=True)
sm.drop(columns="text_len").to_csv(STORIES / "story_map.csv", index=False)

canon = sm.loc[sm["is_canonical"]]
out = df.iloc[canon["row"].to_numpy()][["source", "dt_utc", "title", "text"]]
out = out.reset_index(drop=True)
out.insert(0, "article_id", canon["article_id"].to_numpy())
out["story_id"] = canon["story_id"].to_numpy()
out["story_size"] = canon["story_size"].to_numpy()
out.to_csv(STORIES / "canonical.csv", index=False)

print(f"{STORIES / 'story_map.csv'}: {len(sm)} строк")
print(f"{STORIES / 'canonical.csv'}: {len(out)} строк, правило канона — {CANONICAL}")
out.head(3)

### Артефакты P4

Размеры сюжетов и сжатие корпуса (P5 выполняется один раз на сюжет).

In [ ]:
sizes = sm.drop_duplicates("story_id")["story_size"]
singles = int((sizes == 1).sum())

print(f"сюжетов:   {len(sizes)}")
print(f"из одной статьи:  {singles} ({singles / len(sizes):.1%} сюжетов)")
print(f"крупнейший сюжет: {sizes.max()} статей")
print(f"средний размер:  {sizes.mean():.2f} статьи")
print(f"сжатие корпуса:  {len(df)} -> {len(sizes)} "
      f"(в {len(df) / len(sizes):.2f} раза, снято {1 - len(sizes) / len(df):.1%})")

dist = sizes.value_counts().sort_index().rename("сюжетов").to_frame()
dist["статей"] = dist.index * dist["сюжетов"]
dist.index.name = "размер сюжета"
dist.to_csv(REPORTS / "p4_story_sizes.csv")
print(f"\n{dist.head(10).to_string()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 10+ в один столбец
by_bucket = sizes.clip(upper=10).value_counts().reindex(range(1, 11), fill_value=0)
labels = [str(i) for i in range(1, 10)] + ["10+"]
axes[0].bar(labels, by_bucket.to_numpy(), color=SERIES["semantic"], width=0.6)
axes[0].set_yscale("log")
axes[0].set_title("Сколько сюжетов какого размера")
axes[0].set_xlabel("статей в сюжете")
axes[0].set_ylabel("сюжетов, лог. шкала")

axes[1].hist(semantic["score"].to_numpy(), bins=40, range=(COS_MIN, 1.0),
             color=SERIES["semantic"])
axes[1].set_title("Косинус у семантических пар")
axes[1].set_xlabel("cos")
axes[1].set_ylabel("пар")

plt.tight_layout()
plt.show()

In [ ]:
# примеры сюжетов с наибольшим числом изданий
multi = (sm[sm["story_size"] > 1].groupby("story_id")["source"].nunique()
         .sort_values(ascending=False).head(3).index)

for sid in multi:
    part = sm[sm["story_id"] == sid].sort_values("dt_utc")
    print(f"=== {sid}: {len(part)} статей, изданий {part['source'].nunique()} ===")
    for r in part.itertuples():
        mark = "*" if r.is_canonical else " "
        print(f" {mark} {show(r.row)}")
    print()
print("* — канонический документ сюжета")